In [5]:
import cv2
import numpy as np
import tensorflow as tf

def detect_mask_in_video(input_video_path, output_video_path, trained_model):
    """
    Reads a recorded video, detects faces and masks frame-by-frame, 
    and saves the result as a new video file.
    """
    # 1. Load the pre-trained face detector
    face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
    class_names = ['with_mask', 'without_mask']
    
    # 2. Open the recorded video
    cap = cv2.VideoCapture(input_video_path)
    
    # Check if video opened successfully
    if not cap.isOpened():
        print(f"Error: Could not open video file {input_video_path}")
        return

    # 3. Get video properties to set up the output file
    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    # Define the codec and create a VideoWriter object to save the video
    # 'mp4v' is a standard, widely supported codec for .mp4 files
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_video_path, fourcc, fps, (frame_width, frame_height))
    
    print(f"Processing video... Total frames: {total_frames}")
    frame_count = 0

    # 4. Loop through every frame in the video
    while True:
        ret, frame = cap.read()
        
        # If ret is False, we have reached the end of the video
        if not ret:
            break
            
        frame_count += 1
        
        # Print progress every 50 frames so you know it hasn't frozen
        if frame_count % 50 == 0:
            print(f"Processed {frame_count}/{total_frames} frames...")

        # Convert to grayscale for face detection
        gray_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        faces = face_cascade.detectMultiScale(gray_frame, scaleFactor=1.1, minNeighbors=5, minSize=(60, 60))

        # 5. Process each detected face
        for (x, y, w, h) in faces:
            # Extract and preprocess the face
            face_roi = frame[y:y+h, x:x+w]
            face_resized = cv2.resize(face_roi, (200, 200))
            
            # CRITICAL: Convert BGR to RGB for your model
            face_rgb = cv2.cvtColor(face_resized, cv2.COLOR_BGR2RGB)
            
            # Scale pixels to 0-1 and format for prediction
            face_array = tf.keras.utils.img_to_array(face_rgb) / 255.0
            face_array = np.expand_dims(face_array, axis=0)

            # Make prediction
            predictions = trained_model.predict(face_array, verbose=0)
            predicted_class_idx = np.argmax(predictions, axis=-1)[0]
            confidence = np.max(predictions)
            label = class_names[predicted_class_idx]

            # Draw UI elements
            center_x = x + w // 2
            center_y = y + h // 2
            radius = int((w + h) / 4)
            color = (0, 255, 0) if label == 'with_mask' else (0, 0, 255)
            
            cv2.circle(frame, (center_x, center_y), radius, color, thickness=3)
            cv2.putText(frame, f"{label}: {confidence*100:.1f}%", (x, y - 10), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)

        # 6. Write the processed frame with drawings to the output video
        out.write(frame)

    # 7. Clean up
    cap.release()
    out.release()
    print(f"Finished! Video saved to: {output_video_path}")


# --- How to use it ---
# Ensure your model is loaded first:
my_model = tf.keras.models.load_model('1st_try.keras')
# Run the function:
detect_mask_in_video('cheeti.mp4', 'output_processed.mp4', my_model)

Processing video... Total frames: 545
Processed 50/545 frames...
Processed 100/545 frames...
Processed 150/545 frames...
Processed 200/545 frames...
Processed 250/545 frames...
Processed 300/545 frames...
Processed 350/545 frames...
Processed 400/545 frames...
Processed 450/545 frames...
Processed 500/545 frames...
Finished! Video saved to: output_processed.mp4
